In [1]:
%matplotlib inline

In [2]:
import numpy as np
import seaborn as sns
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import sys
import pyreadstat
import requests
import time
import urllib.request
import os
import gtfs_kit as gk
import zipfile
import statsmodels
import json

In [36]:
# =============================================================================
# LONDON CONFIGURATION BLOCK
# Source: NTS Data Extract User Guide + Main Table Variables London.csv
# All codes verified via value inspection (see cells below).
# =============================================================================

# ---- File paths ----
nts_files = {
    "trip": r"C:\Users\vladimir.koev\OneDrive - Трансметрикс АД\SoftUni\Data Science\Final Exam\data\raw\london\raw\UKDA-5340-tab\tab\trip_eul_2002-2024.tab",
    "individual": r"C:\Users\vladimir.koev\OneDrive - Трансметрикс АД\SoftUni\Data Science\Final Exam\data\raw\london\raw\UKDA-5340-tab\tab\individual_eul_2002-2024.tab",
    "household": r"C:\Users\vladimir.koev\OneDrive - Трансметрикс АД\SoftUni\Data Science\Final Exam\data\raw\london\raw\UKDA-5340-tab\tab\household_eul_2002-2024.tab",
    "psu": r"C:\Users\vladimir.koev\OneDrive - Трансметрикс АД\SoftUni\Data Science\Final Exam\data\raw\london\raw\UKDA-5340-tab\tab\psu_eul_2002-2024.tab",
    "day": r"C:\Users\vladimir.koev\OneDrive - Трансметрикс АД\SoftUni\Data Science\Final Exam\data\raw\london\raw\UKDA-5340-tab\tab\day_eul_2002-2024.tab",
}

# ---- Year filter (applied at load time for memory efficiency) ----
survey_year = 2023

# ---- Geographic filter ----
# PSUStatsReg_B01ID: Statistical Region
# Inner London code — to be verified in inspection cell
inner_london_code = 7  

# ---- Trip purpose filter (TripPurpose_B04ID, 8 categories) ----
# 1=Commuting, 2=Business, 3=Education, 4=Escort,
# 5=Shopping, 6=Other personal, 7=Leisure, 8=Other
commute_code_london = 1

# ---- Mode codes (MainMode_B04ID, 13 categories) ----
# NTS publication table standard breakdown:
#  1=Walk, 2=Bicycle, 3=Car/van driver, 4=Car/van passenger,
#  5=Motorcycle, 6=Other private, 7=Bus in London,
#  8=Other local bus, 9=Non-local bus/coach,
# 10=London Underground/metro/light rail, 11=Surface Rail,
# 12=Taxi/minicab, 13=Other public transport
mode_car_driver  = 3
mode_bicycle     = 2
mode_bus_london  = 7
mode_bus_other   = 8
mode_underground = 10 # Underground + DLR + light rail + tram
mode_other_pt = 13 

# Choice set: car driver, PT (bus + underground), bicycle
pt_codes_london  = [mode_bus_london, mode_bus_other, mode_underground, mode_other_pt]
choice_modes_london = [mode_car_driver] + pt_codes_london + [mode_bicycle]

# ---- Weekday filter (TravelWeekDay_B01ID) ----
weekday_codes_london = [1, 2, 3, 4, 5]   # Mon–Fri

# ---- Holiday filter (TravelDayType_B01ID, from 2008 onwards) ----
# 1 = Normal day, 2+ = bank holiday / special day
normal_day_code = 1

# ---- Socioeconomic missing codes (NTS convention) ----
# -8 = "Don't know", -9 = "Not answered/applicable"
missing_codes = [-8, -9, -10]

# ---- Driving licence (DrivLic_B02ID, 3 categories) ----
# 1 = Full licence, 2 = Provisional, 3 = No licence
full_licence_code = 1

# ---- Income variable (year-specific quintiles for England) ----
income_var_2023 = "HHIncQIS2023Eng_B01ID"  # interview-sample quintiles

# ---- Purpose labels (for EDA bar chart) ----
purpose_labels = {
    1: "Commuting", 2: "Business", 3: "Education", 4: "Escort",
    5: "Shopping", 6: "Other personal", 7: "Leisure", 8: "Other/walk"
}

# ---- Mode labels (for EDA charts, all 13 categories) ----
mode_labels_b04 = {
     1: "Walk",  2: "Bicycle",  3: "Car driver",  4: "Car passenger",
     5: "Motorcycle",  6: "Other private",  7: "Bus (London)",
     8: "Other local bus",  9: "Non-local bus/coach",
    10: "Underground/metro/LR", 11: "Surface rail",
    12: "Taxi/minicab", 13: "Other public"
}

print("London config loaded.")
print(f"  Survey year : {survey_year}")
print(f"  PT codes    : {pt_codes_london}")
print(f"  Choice set  : car({mode_car_driver}), PT{pt_codes_london}, bike({mode_bicycle})")

London config loaded.
  Survey year : 2023
  PT codes    : [7, 8, 10, 13]
  Choice set  : car(3), PT[7, 8, 10, 13], bike(2)


## 1. Data Source Declaration

As a second datasource we are going to use the UK National Travel Survey (NTS) for 2023. This section is going to mirrors the Amsterdam pipeline: load → inspect → filter → rename → validate.

The target output is a `london_processed` dataframe with the same 19 shared-schema columns as in amsterdams data.

**Data source:** UK National Travel Survey (NTS), End User Licence dataset, 2002–2024 (Department for Transport, 2024).
We restrict to `SurveyYear == 2023` to match the Amsterdam ODiN 2023 sample.

## 1. Data Source Loading

In [37]:
# Load PSU table (small) and get 2023 PSU IDs
psu_raw = pd.read_csv(nts_files["psu"], sep="\t", low_memory=False)

In [38]:
psu_raw.head(5)

,PSUID,SurveyYear,SurveyYear_B01ID,PSUGOR_B02ID,PSUStatsReg_B01ID,PSUCountry_B01ID
0,2002000167,2002,8,11,15,3
1,2002000166,2002,8,11,15,3
2,2002000395,2002,8,11,15,3
3,2002000653,2002,8,6,7,1
4,2002000083,2002,8,6,7,1


In [39]:
psu_2023 = psu_raw[psu_raw.SurveyYear == survey_year].copy() # Get the psu_data for 2023 only
psu_ids_2023 = set(psu_2023.PSUID)

In [40]:
psu_2023.head(5)

,PSUID,SurveyYear,SurveyYear_B01ID,PSUGOR_B02ID,PSUStatsReg_B01ID,PSUCountry_B01ID
14959,2023000061,2023,29,-10,10,-10
14960,2023000304,2023,29,-10,11,-10
14961,2023000125,2023,29,-10,12,-10
14962,2023000362,2023,29,-10,8,-10
14963,2023000078,2023,29,-10,12,-10


In [41]:
psu_2023.shape

(902, 6)

In [42]:
# Load remaining tables and filter to 2023 PSUIDs
def load_and_filter(filepath, psu_ids, label):
    """Load tab file, keep rows matching 2023 PSUIDs."""
    dataset = pd.read_csv(filepath, sep="\t", low_memory = False)
    n_total = len(dataset)
    dataset = dataset[dataset.PSUID.isin(psu_ids)].copy()

    print(f"{label}: {n_total:,} total → {len(dataset):,} rows for {survey_year}")

    return dataset

household_2023 = load_and_filter(nts_files["household"], psu_ids_2023, "Household")
individual_2023 = load_and_filter(nts_files["individual"], psu_ids_2023, "Individual")
day_2023 = load_and_filter(nts_files["day"], psu_ids_2023, "Day")
trip_2023 = load_and_filter(nts_files["trip"], psu_ids_2023, "Trip")

# --- 1c. Save 20-row samples for inspection ---
for name, df in [("trip", trip_2023), ("individual", individual_2023),
                  ("household", household_2023), ("psu", psu_2023), ("day", day_2023)]:
    df.head(20).to_csv(f"london_sample_{name}_2023.csv", index=False)
    
print(f"\nSample CSVs saved (20 rows each).")

Household: 178,153 total → 7,427 rows for 2023
Individual: 421,210 total → 16,822 rows for 2023
Day: 2,735,894 total → 98,679 rows for 2023
Trip: 5,577,192 total → 192,848 rows for 2023

Sample CSVs saved (20 rows each).
